# 02 — Sleep Spindle and Slow Oscillation Detection

**Purpose:** Demonstrate YASA spindle and slow oscillation detection.
These events are the targets for TMR cue delivery.

**Spindles:** 12-15 Hz bursts in N2/N3, facilitate memory consolidation.
**Slow oscillations:** <1 Hz waves in N3, coupling with spindles = optimal consolidation.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
import mne
plt.style.use('dark_background')

from classifiers.tmr.event_detection import (
    EventDetectionConfig, detect_spindles, detect_slow_oscillations
)

## 1. Detect Spindles

In [ ]:
# Create synthetic N2 sleep data with spindles
sfreq = 256
duration = 60  # 1 minute
n_samples = sfreq * duration
times = np.arange(n_samples) / sfreq
rng = np.random.RandomState(42)

# Background: delta + noise
signal = 30e-6 * np.sin(2 * np.pi * 1.5 * times)  # slow wave
signal += 5e-6 * rng.randn(n_samples)  # noise

# Add spindle bursts at random times
for t in [10, 25, 38, 52]:
    start = int(t * sfreq)
    duration_sp = int(0.8 * sfreq)
    envelope = np.hanning(duration_sp)
    spindle = 15e-6 * envelope * np.sin(2 * np.pi * 13.5 * np.arange(duration_sp) / sfreq)
    signal[start:start+duration_sp] += spindle

info = mne.create_info(['C4'], sfreq=sfreq, ch_types='eeg')
raw = mne.io.RawArray(signal[np.newaxis, :], info, verbose=False)

# Detect
try:
    sp_result = detect_spindles(raw)
    print(f'Spindles found: {sp_result.n_spindles}')
    print(f'Density: {sp_result.density:.1f}/min')
    if sp_result.n_spindles > 0:
        print(f'Mean frequency: {sp_result.mean_frequency:.1f} Hz')
        print(f'Mean duration: {sp_result.mean_duration:.2f} s')
except Exception as e:
    print(f'Detection note: {e}')